# Step 1 — Fetch Movie Data from the TMDB API

**Goal:** call the TMDB API for a list of movie ids, look at what the raw data
actually looks like, and save it untouched to `data/raw/`.

This notebook only *extracts*. All cleaning happens in notebook 02 — separating
the two means we hit the API once and can re-run the cleaning as often as we like.

## 1. Setup
<!-- 
The API key lives in a `.env` file (git-ignored) so it never ends up on GitHub.
`load_dotenv()` reads it into the environment, and we grab it with `os.getenv`.

The `sys.path.append("..")` line lets this notebook (which lives in `notebooks/`)
import from `src/` in the project root. -->

In [1]:
import os
import sys

import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.append("..")
from src.tmdb_client import fetch_movie, fetch_movies, save_raw

load_dotenv("../.env")
API_KEY = os.getenv("TMDB_API_KEY")

print("API key loaded ✔")

API key loaded ✔


## 2. Explore ONE movie first

Before fetching everything, lets look at a single raw response — to have a fair idea of the data. We'll fetch *Avengers: Endgame* (id `299534`) and
inspect its structure.

In [ ]:
sample = fetch_movie(299534, API_KEY)

# print(sample)
print(type(sample))
print(f"{len(sample)} top-level fields:\n")
print(sorted(sample.keys()))

<class 'dict'>
28 top-level fields:

['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'credits', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


Some fields are simple scalars (`title`, `budget`, `runtime`), but others are
**nested** — lists of dictionaries sitting inside one field. These are the ones
We'll flatten latter.

In [3]:
print("genres:", sample["genres"])
print()
print("belongs_to_collection:", sample["belongs_to_collection"])
print()
print("production_countries:", sample["production_countries"])

genres: [{'id': 12, 'name': 'Adventure'}, {'id': 878, 'name': 'Science Fiction'}, {'id': 28, 'name': 'Action'}]

belongs_to_collection: {'id': 86311, 'name': 'The Avengers Collection', 'poster_path': '/yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg', 'backdrop_path': '/2UNUv4NJdC36E5myDHACBJ99EwL.jpg'}

production_countries: [{'iso_3166_1': 'US', 'name': 'United States of America'}]


In [4]:
# Because we asked for append_to_response=credits, cast & crew came along too.
print(f"cast size: {len(sample['credits']['cast'])}")
print(f"crew size: {len(sample['credits']['crew'])}")
print()
print("first cast entry:", sample["credits"]["cast"][0])

cast size: 105
crew size: 613

first cast entry: {'adult': False, 'gender': 2, 'id': 3223, 'known_for_department': 'Acting', 'name': 'Robert Downey Jr.', 'original_name': 'Robert Downey Jr.', 'popularity': 5.7628, 'profile_path': '/5qHNjhtjMD4YWH3UP0rm4tKwxCL.jpg', 'cast_id': 493, 'character': 'Tony Stark / Iron Man', 'credit_id': '5e85cd735294e700134abf26', 'order': 0}


## 3. Issue with id '0'

The project's id list deliberately starts with `0`, which doesn't exist. The API
answers with **HTTP 404**. Our client treats that as "skip it" instead of crashing —
handling failures gracefully is a core part of any extraction pipeline.

In [5]:
raw_response = requests.get(
    "https://api.themoviedb.org/3/movie/0",
    params={"api_key": API_KEY},
    timeout=10,
)
print("status code:", raw_response.status_code)
print(raw_response.json())

print("\nfetch_movie returns:", fetch_movie(0, API_KEY))

status code: 404
{'success': False, 'status_code': 6, 'status_message': 'Invalid id: The pre-requisite id is invalid or not found.'}

fetch_movie returns: None


## 4. Fetch all movies

Now the real extraction — the full id list from the project brief.

In [6]:
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513,
]

movies = fetch_movies(movie_ids, API_KEY)
print(f"\nFetched {len(movies)} of {len(movie_ids)} ids")

id 0: not found, skipped
id 299534: Avengers: Endgame
id 19995: Avatar
id 140607: Star Wars: The Force Awakens
id 299536: Avengers: Infinity War
id 597: Titanic
id 135397: Jurassic World
id 420818: The Lion King
id 24428: The Avengers
id 168259: Furious 7
id 99861: Avengers: Age of Ultron
id 284054: Black Panther
id 12445: Harry Potter and the Deathly Hallows: Part 2
id 181808: Star Wars: The Last Jedi
id 330457: Frozen II
id 351286: Jurassic World: Fallen Kingdom
id 109445: Frozen
id 321612: Beauty and the Beast
id 260513: Incredibles 2

Fetched 18 of 19 ids


## 5. Save the raw data — untouched

We save as **JSON**, not CSV. A CSV would silently turn the nested fields into
strings and lose structure; JSON preserves the responses exactly as the API sent
them. This is our immutable source of truth — notebook 02 reads from here.

In [7]:
save_raw(movies, "../data/raw/movies_raw.json")

Saved 18 movies to ../data/raw/movies_raw.json


## 6. First look as a DataFrame

`pd.DataFrame(list_of_dicts)` turns each dict into a row and each key into a
column. This preview is just to see what we're dealing with — the real
DataFrame work starts in notebook 02.

In [8]:
df = pd.DataFrame(movies)
print(df.shape)
df.head(3)

(18, 28)


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.239,28036,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.609,34232,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
2,False,/k6EOrckWFuz7I4z4wiRwz8zsj4H.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,136,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.248,20661,"{'cast': [{'adult': False, 'gender': 1, 'id': ..."


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  18 non-null     bool   
 1   backdrop_path          18 non-null     object 
 2   belongs_to_collection  16 non-null     object 
 3   budget                 18 non-null     int64  
 4   genres                 18 non-null     object 
 5   homepage               18 non-null     object 
 6   id                     18 non-null     int64  
 7   imdb_id                18 non-null     object 
 8   origin_country         18 non-null     object 
 9   original_language      18 non-null     object 
 10  original_title         18 non-null     object 
 11  overview               18 non-null     object 
 12  popularity             18 non-null     float64
 13  poster_path            18 non-null     object 
 14  production_companies   18 non-null     object 
 15  producti

 `.info()` as a quick glance on the structure



 **Next: `02_data_cleaning.ipynb`**